## Conv1d and Conv2d

In [270]:
import torch

In [271]:
x = torch.randn(8, 10, 1000, dtype=torch.float64)
a = torch.nn.Conv1d(10, 20, 7, bias=False, dtype=x.dtype)
b = torch.nn.Conv2d(1, 20, (10, 7), bias=False, dtype=x.dtype)
a.weight.data = b.weight.data.squeeze()
ya = a(x)
yb = b(x[:, None, ...]).squeeze()

In [272]:
torch.allclose(ya, yb)

True

In [273]:
ya.shape, yb.shape

(torch.Size([8, 20, 994]), torch.Size([8, 20, 994]))

## PCA

In [274]:
import numpy as np
import scipy as sp
from sklearn.decomposition import PCA

np.random.seed(0)

n_features = 500
tmp = np.random.randint(1, 5, (1, 500))
X_train = np.random.randn(10000, 500) * tmp
X_test = np.random.randn(3000, 500) * tmp

r = 10
pca = PCA(n_components=r, whiten=False, svd_solver='full', random_state=0)
pca.fit(X_train)

Y_train = pca.transform(X_train)
Y_test = pca.transform(X_test)

# Compute the manual PCA using SVD on the centered data
mean = X_train.mean(axis=0)
Z_train = X_train - mean
Z_test = X_test - mean

U, S, Vt = np.linalg.svd(Z_train, full_matrices=False)
manual_components = Vt[:r, :]

# Align the sign of each component to match sklearn's PCA components
for i in range(r):
    if np.dot(manual_components[i], pca.components_[i]) < 0:
        manual_components[i] *= -1

# Now, transform using the aligned manual components
R_train = Z_train @ manual_components.T
R_test = Z_test @ manual_components.T

print(np.allclose(R_train, Y_train))
print(np.allclose(R_test, Y_test))

True
True


In [275]:
pca.inverse_transform(Y_train).shape

(10000, 500)